In [ ]:
# a family relational graphRAG to ask queries regarding how people are related

In [ ]:
print(9*9)

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import tool

from pydantic import BaseModel, Field

In [ ]:
class Triads(BaseModel):
    list_of_triplets: list[list[str]] = Field(description = "a list of all the relational triplets")

In [ ]:

# 1. BUILD THE KNOWLEDGE GRAPH (The "Index")

def get_triads_from_text(input_text:  str) -> list[list[str]]:
    extraction_prompt = f"""
    Extract a Knowledge Graph from the input text below. 
    Rules:
    1. Identify Entities (Nodes) and their Relationships (Edges).
    2. NORMALIZE: Convert all entities to title case (e.g., 'even numbers' -> 'Even Numbers').
    3. OUTPUT: Provide a list of triples in the format: [["Subject", "Relationship", "Object"]].

    Input Text: {input_text}
    """

    llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")
    llm_structured = llm.with_structured_output(Triads)
    response = llm_structured.invoke(input=extraction_prompt)
    triplets = response.list_of_triplets

    return triplets

In [ ]:
relation_text = """
Arthur is the patriarch of the Sterling family and is married to Martha. They have two children: a son named Bob and a daughter named Diana. 
Diana eventually married a man named Edward, and together they had a daughter named Alice. Bob, who never married, is known for being the favorite sibling of Diana. 
During the holidays, Alice always looks forward to seeing her mother's brother, as he tells the best stories.
"""
triplets = get_triads_from_text(relation_text)

kg = nx.MultiDiGraph()
for subject, relation, obj in triplets:
    kg.add_edge(subject, obj, relation=relation)

In [ ]:
# kg.neighbors("Alice")

In [ ]:
# results = []
# for neighbor in kg.neighbors("Arthur"):
#     print(neighbor)
    
#     edge_data = kg.get_edge_data("Arthur", neighbor)
#     relation = edge_data[0]['relation']
#     results.append(f"Arthur {relation} {neighbor}")
# results

In [ ]:
# 2. DEFINE THE GRAPH TOOL
@tool
def graph_lookup(entity_query: str):
    """Consult the Knowledge Graph to see relationships for a specific entity.
    Look up a SINGLE entity in the knowledge graph to find its relations.
    If you need to lookup multiple entities, make separate tool calls for each entity"""
    
    entity_query = entity_query.title()

    if entity_query not in kg:
        return f"None {entity_query} not found in graph."
    
    results = []
    for neighbor in kg.neighbors(entity_query):
        edge_data = kg.get_edge_data(entity_query, neighbor)
        relation = edge_data[0]['relation']
        results.append(f"{entity_query} {relation} {neighbor}")
    
    return "\n".join(results) if results else "No direct relationships found."

In [ ]:
# 3. THE AGENT LOOP
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash") # Or your preferred version
llm_with_tools = llm.bind_tools([graph_lookup])

query = "Based on the family records, how is Bob related to Alice?"
ai_msg = llm_with_tools.invoke(query)
print(f"AI Choice: {ai_msg.tool_calls}")

In [ ]:
if ai_msg.tool_calls:
    entities_to_lookup = [tool["args"]["entity_query"]  for tool in ai_msg.tool_calls]
    context_each_entity = [graph_lookup.invoke(entity) for entity in entities_to_lookup]
    context = "\n".join(context_each_entity)
    print(entities_to_lookup)
else:
    context = "The AI didn't identify any entities to look in the graph"

In [ ]:
final_response = llm.invoke(f"Context: {context}\n\nQuestion: {query}") 

print("\n--- FINAL ANSWER ---")
print(final_response.content)

In [ ]:
print(context_each_entity)

### Visualize the graph

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

def visualize_knowledge_graph(graph):
    plt.figure(figsize=(12, 8))
    
    # spring_layout helps spread out nodes so arrows are visible
    pos = nx.spring_layout(graph, k=1.5, seed=42) 
    
    # 1. Draw the Nodes
    nx.draw_networkx_nodes(graph, pos, node_color='skyblue', node_size=2500)
    
    # 2. Draw the Edges with Arrows
    nx.draw_networkx_edges(
        graph, 
        pos, 
        arrowstyle='-|>', 
        arrowsize=20, 
        edge_color='gray',
        width=2,
        node_size=2500, # ADD THIS: Tells the edge where to stop
        connectionstyle='arc3,rad=0.1' # Small radius helps if arrows overlap
    )
    
    # 3. Draw the Node Labels
    nx.draw_networkx_labels(graph, pos, font_size=10, font_weight='bold')
    
    # 4. Draw the Relationship Labels (The "Edges")
    edge_labels = {(u, v): d['relation'] for u, v, d in graph.edges(data=True)}
    nx.draw_networkx_edge_labels(graph, pos, edge_labels=edge_labels, font_color='red', font_size=8)
    
    plt.title("Arthur's family tree")
    plt.axis('off') # Hide the grid/axis
    plt.show()

# Run the visualization
visualize_knowledge_graph(kg)